In [ ]:
# gold.py
from datetime import datetime
from minio import Minio
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum as spark_sum, avg, when, lit, 
    first, coalesce, abs as spark_abs, count, countDistinct
)
from pyspark.sql.types import DoubleType, IntegerType, StringType, BooleanType
from config import MinIOConfig
import logging
from typing import List, Dict, Optional
import time

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class GoldProcessor:
    """Classe base para processar dados da camada Silver → Gold"""
    
    def __init__(self, minio_config: MinIOConfig = None):
        # Guardar as informações
        self.minio_config = minio_config or MinIOConfig()
        # Abrir a conexão de fato
        self.minio_client = Minio(
            self.minio_config.endpoint,
            access_key=self.minio_config.access_key,
            secret_key=self.minio_config.secret_key,
            secure=self.minio_config.secure
        )
        self.spark = self._create_spark_session()
        self._ensure_bucket_exists()
        logger.info("✅ GoldProcessor inicializado")
    
    def _create_spark_session(self) -> SparkSession:
        """Cria sessão Spark com Delta Lake e S3 configurados"""
        
        # Configurar Spark Builder
        builder = (
            # Inicia o construtor da sessão Spark
            SparkSession.builder

                # Nome da aplicação exibida na interface do Spark
                .appName("CVM_Gold_Processing")

                # ------------------------------------------------------------
                # Dependências externas que o Spark precisa baixar automaticamente
                # ------------------------------------------------------------
                .config(
                    "spark.jars.packages",
                    "io.delta:delta-core_2.12:2.4.0"  # Só Delta
                )
                .config(
                    "spark.jars",
                    "/opt/spark-jars/hadoop-aws-3.3.4.jar,/opt/spark-jars/aws-java-sdk-bundle-1.12.262.jar"
                )
                .config(
                    "spark.driver.extraClassPath",
                    "/opt/spark-jars/hadoop-aws-3.3.4.jar:/opt/spark-jars/aws-java-sdk-bundle-1.12.262.jar"
                )
                .config(
                    "spark.executor.extraClassPath",
                    "/opt/spark-jars/hadoop-aws-3.3.4.jar:/opt/spark-jars/aws-java-sdk-bundle-1.12.262.jar"
                )

                # ------------------------------------------------------------
                # Ativação do Delta Lake no Spark
                # ------------------------------------------------------------

                # Liga extensões SQL específicas do Delta Lake
                .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")

                # Torna o catálogo Delta o padrão para criação/leitura de tabelas
                .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

                # ------------------------------------------------------------
                # Configurações para acessar o MinIO usando S3A
                # ------------------------------------------------------------

                # Endereço HTTP do serviço MinIO
                .config("spark.hadoop.fs.s3a.endpoint", f"http://{self.minio_config.endpoint}")

                # Access key (usuário público) do MinIO
                .config("spark.hadoop.fs.s3a.access.key", self.minio_config.access_key)

                # Secret key (senha) do MinIO
                .config("spark.hadoop.fs.s3a.secret.key", self.minio_config.secret_key)

                # Obrigatório para MinIO: usa estilo path, não VHost
                .config("spark.hadoop.fs.s3a.path.style.access", "true")

                # Define explicitamente o filesystem S3A correto
                .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

                # Desativa SSL porque MinIO normalmente roda em HTTP no ambiente local
                .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

                # Usa provedor simples de credenciais (ideal para MinIO)
                .config("spark.hadoop.fs.s3a.aws.credentials.provider",
                        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

                # ------------------------------------------------------------
                # Configurações de performance do Spark
                # ------------------------------------------------------------

                # Ativa otimização adaptativa → Spark ajusta planos de execução dinamicamente
                .config("spark.sql.adaptive.enabled", "true")

                # Número de partições usadas em operações como join e groupBy
                .config("spark.sql.shuffle.partitions", "8")

                # Tamanho máximo de cada partição de arquivo (128 MB)
                # Evita gerar muitos arquivos pequenos
                .config("spark.sql.files.maxPartitionBytes", 134217728)
        )
        
        # Adicionar Delta Lake
        spark = configure_spark_with_delta_pip(builder).getOrCreate()
        spark.sparkContext.setLogLevel("WARN")
        
        logger.info("✅ Spark Session criada com Delta Lake")
        return spark
    
    def _ensure_bucket_exists(self):
        """Garante que o bucket gold existe"""
        if not self.minio_client.bucket_exists(self.minio_config.bucket_gold):
            self.minio_client.make_bucket(self.minio_config.bucket_gold)
            logger.info(f"✅ Bucket {self.minio_config.bucket_gold} criado")
        else:
            logger.info(f"✅ Bucket {self.minio_config.bucket_gold} já existe")


class CVMGoldProcessor(GoldProcessor):
    """Processa dados da CVM da camada Silver para Gold - Indicadores de Graham"""
    
    # Tabelas Silver necessárias
    SILVER_TABLES = ["DRE_con", "BPA_con", "BPP_con"]
    
    def process_year(self, ano: int) -> bool:
        """
        Processa dados de um ano específico e calcula indicadores de Graham
        
        Args:
            ano: Ano a processar (ex: 2024)
            
        Returns:
            True se processamento foi bem-sucedido
        """
        try:
            start_time = time.time()
            logger.info(f"🎯 Iniciando processamento Gold para ano {ano}")
            
            # 1. Carregar tabelas Silver
            logger.info("📚 Carregando tabelas Silver...")
            dre = self._read_silver_table("DRE_con", ano)
            bpa = self._read_silver_table("BPA_con", ano)
            bpp = self._read_silver_table("BPP_con", ano)
            
            # Validar se há dados
            if dre.count() == 0:
                logger.warning(f"⚠️  Nenhum dado DRE encontrado para {ano}")
                return False
            
            # 2. Extrair indicadores
            logger.info("🔍 Extraindo indicadores...")
            lucro = self._extract_lucro_liquido(dre)
            patrimonio = self._extract_patrimonio_liquido(bpp)
            ativo = self._extract_ativo_circulante(bpa)
            passivo = self._extract_passivo_circulante(bpp)
            dividendos = self._extract_dividendos(dre)
            num_acoes = self._extract_num_acoes(dre)
            
            # 3. Consolidar indicadores
            logger.info("🔗 Consolidando indicadores...")
            indicadores = self._consolidate_indicators(
                lucro, patrimonio, ativo, passivo, dividendos, num_acoes, ano
            )
            
            # 4. Calcular métricas derivadas
            logger.info("🧮 Calculando métricas derivadas...")
            indicadores = self._calculate_derived_metrics(indicadores)
            
            # 5. Aplicar filtros de Graham
            logger.info("✅ Aplicando critérios de Benjamin Graham...")
            indicadores = self._apply_graham_filters(indicadores)
            
            # 6. Salvar na camada Gold
            logger.info("💾 Salvando na camada Gold...")
            self._save_to_gold(indicadores, ano)
            
            # 7. Exibir estatísticas
            self._show_statistics(indicadores, ano)
            
            elapsed = time.time() - start_time
            logger.info(f"⏱️  Processamento concluído em {elapsed:.2f}s")
            
            return True
            
        except Exception as e:
            logger.error(f"❌ Erro ao processar ano {ano}: {e}")
            import traceback
            traceback.print_exc()
            return False
    
    def _read_silver_table(self, table_name: str, ano: int):
        """
        Lê uma tabela Delta da camada Silver
        
        Args:
            table_name: Nome da tabela (DRE_con, BPA_con, BPP_con)
            ano: Ano de referência
            
        Returns:
            Spark DataFrame
        """
        delta_path = f"s3a://{self.minio_config.bucket_silver}/cvm/{table_name}"
        
        logger.info(f"  📖 Lendo {table_name} de {delta_path}")
        
        df = self.spark.read.format("delta").load(delta_path)
        
        # Filtrar por ano
        df = df.filter(
            (col("ano_referencia") == ano) | 
            (col("ano_referencia") == str(ano))
        )
        
        count = df.count()
        logger.info(f"     ✓ {count:,} registros carregados")
        
        return df
    
    def _extract_lucro_liquido(self, dre_df):
        """
        Extrai Lucro Líquido da DRE
        
        CRITÉRIO GRAHAM: Empresa não teve prejuízo
        """
        logger.info("  💰 Extraindo Lucro Líquido...")
        
        lucro = dre_df.filter(
            col("DS_CONTA").rlike("(?i)lucro.*líquido")
        ).groupBy("CNPJ_CIA", "DENOM_CIA", "DT_REFER").agg(
            spark_sum("VL_CONTA").alias("lucro_liquido")
        )
        
        count = lucro.count()
        logger.info(f"     ✓ {count:,} registros extraídos")
        return lucro
    
    def _extract_patrimonio_liquido(self, bpp_df):
        """
        Extrai Patrimônio Líquido do BPP
        
        CRITÉRIO GRAHAM: P/VPA (necessita PL para calcular VPA)
        """
        logger.info("  🏦 Extraindo Patrimônio Líquido...")
        
        patrimonio = bpp_df.filter(
            col("DS_CONTA").rlike("(?i)patrimônio.*líquido")
        ).groupBy("CNPJ_CIA", "DT_REFER").agg(
            spark_sum("VL_CONTA").alias("patrimonio_liquido")
        )
        
        count = patrimonio.count()
        logger.info(f"     ✓ {count:,} registros extraídos")
        return patrimonio
    
    def _extract_ativo_circulante(self, bpa_df):
        """
        Extrai Ativo Circulante do BPA
        
        CRITÉRIO GRAHAM: Liquidez Corrente >= 1.0
        """
        logger.info("  📊 Extraindo Ativo Circulante...")
        
        ativo = bpa_df.filter(
            col("DS_CONTA").rlike("(?i)ativo.*circulante")
        ).groupBy("CNPJ_CIA", "DT_REFER").agg(
            spark_sum("VL_CONTA").alias("ativo_circulante")
        )
        
        count = ativo.count()
        logger.info(f"     ✓ {count:,} registros extraídos")
        return ativo
    
    def _extract_passivo_circulante(self, bpp_df):
        """
        Extrai Passivo Circulante do BPP
        
        CRITÉRIO GRAHAM: Liquidez Corrente >= 1.0
        """
        logger.info("  📉 Extraindo Passivo Circulante...")
        
        passivo = bpp_df.filter(
            col("DS_CONTA").rlike("(?i)passivo.*circulante")
        ).groupBy("CNPJ_CIA", "DT_REFER").agg(
            spark_sum("VL_CONTA").alias("passivo_circulante")
        )
        
        count = passivo.count()
        logger.info(f"     ✓ {count:,} registros extraídos")
        return passivo
    
    def _extract_dividendos(self, dre_df):
        """
        Extrai Dividendos da DRE
        
        CRITÉRIO GRAHAM: Empresa paga dividendos
        """
        logger.info("  💸 Extraindo Dividendos...")
        
        dividendos = dre_df.filter(
            col("DS_CONTA").rlike("(?i)dividendo")
        ).groupBy("CNPJ_CIA", "DT_REFER").agg(
            spark_sum(spark_abs(col("VL_CONTA"))).alias("dividendos")
        )
        
        count = dividendos.count()
        logger.info(f"     ✓ {count:,} registros extraídos")
        return dividendos
    
    def _extract_num_acoes(self, dre_df):
        """
        Extrai número de ações da DRE (ou usa estimativa)
        
        Necessário para calcular LPA e VPA
        """
        logger.info("  🔢 Extraindo número de ações...")
        
        # Tentar buscar LPA reportado
        lpa_info = dre_df.filter(
            col("DS_CONTA").rlike("(?i)lucro.*por.*ação")
        ).groupBy("CNPJ_CIA", "DT_REFER").agg(
            first("VL_CONTA").alias("lpa_reportado")
        )
        
        # Se não tiver LPA, usar estimativa conservadora de 1 bilhão de ações
        num_acoes = lpa_info.withColumn(
            "num_acoes",
            when(col("lpa_reportado").isNotNull(), 
                 lit(1000000000))
            .otherwise(lit(1000000000))
        ).select("CNPJ_CIA", "DT_REFER", "num_acoes")
        
        count = num_acoes.count()
        logger.info(f"     ✓ {count:,} registros (estimativa: 1bi ações)")
        return num_acoes
    
    def _consolidate_indicators(self, lucro, patrimonio, ativo, passivo, 
                                dividendos, num_acoes, ano):
        """
        Consolida todos os indicadores em um único DataFrame
        """
        # Base: lucro (tem DENOM_CIA)
        indicadores = lucro
        
        # Join com outros indicadores
        indicadores = indicadores \
            .join(patrimonio, ["CNPJ_CIA", "DT_REFER"], "left") \
            .join(ativo, ["CNPJ_CIA", "DT_REFER"], "left") \
            .join(passivo, ["CNPJ_CIA", "DT_REFER"], "left") \
            .join(dividendos, ["CNPJ_CIA", "DT_REFER"], "left") \
            .join(num_acoes, ["CNPJ_CIA", "DT_REFER"], "left")
        
        # Adicionar ano de referência
        indicadores = indicadores.withColumn("ANO_REFERENCIA", lit(ano))
        
        # Adicionar trimestre baseado na data
        indicadores = indicadores.withColumn(
            "TRIMESTRE", 
            when(col("DT_REFER").endswith("03-31"), lit("T1"))
            .when(col("DT_REFER").endswith("06-30"), lit("T2"))
            .when(col("DT_REFER").endswith("09-30"), lit("T3"))
            .when(col("DT_REFER").endswith("12-31"), lit("T4"))
            .otherwise(lit("ANUAL"))
        )
        
        count = indicadores.count()
        logger.info(f"  ✓ {count:,} registros consolidados")
        
        return indicadores
    
    def _calculate_derived_metrics(self, df):
        """
        Calcula métricas derivadas: LPA, VPA, LC
        
        LPA = Lucro Líquido / Número de Ações
        VPA = Patrimônio Líquido / Número de Ações
        LC  = Ativo Circulante / Passivo Circulante
        """
        df = df.withColumn(
            "lpa",
            when(col("num_acoes").isNotNull() & (col("num_acoes") > 0),
                 col("lucro_liquido") / col("num_acoes"))
            .otherwise(lit(None).cast(DoubleType()))
        ).withColumn(
            "vpa",
            when(col("num_acoes").isNotNull() & (col("num_acoes") > 0),
                 col("patrimonio_liquido") / col("num_acoes"))
            .otherwise(lit(None).cast(DoubleType()))
        ).withColumn(
            "lc",
            when(col("passivo_circulante").isNotNull() & (col("passivo_circulante") != 0),
                 col("ativo_circulante") / col("passivo_circulante"))
            .otherwise(lit(None).cast(DoubleType()))
        )
        
        logger.info("  ✓ Métricas derivadas calculadas (LPA, VPA, LC)")
        
        return df
    
    def _apply_graham_filters(self, df):
        """
        Aplica os critérios de filtro de Benjamin Graham
        
        FILTRO 1: Lucro Líquido > 0 (empresa não teve prejuízo)
        FILTRO 2: P/L × P/VPA < 22.5 (requer preços - fase 2)
        FILTRO 3: Dividendos > 0 (empresa paga dividendos)
        FILTRO 4: LC >= 1.0 (boa saúde financeira)
        """
        df = df.withColumn(
            "passou_filtro_1",
            when(col("lucro_liquido") > 0, True).otherwise(False)
        ).withColumn(
            "passou_filtro_3",
            when(coalesce(col("dividendos"), lit(0)) > 0, True).otherwise(False)
        ).withColumn(
            "passou_filtro_4",
            when(coalesce(col("lc"), lit(0)) >= 1.0, True).otherwise(False)
        ).withColumn(
            "passou_todos_filtros",
            when(
                (col("passou_filtro_1") == True) & 
                (col("passou_filtro_3") == True) & 
                (col("passou_filtro_4") == True),
                True
            ).otherwise(False)
        )
        
        logger.info("  ✓ Filtros de Graham aplicados")
        
        return df
    
    def _save_to_gold(self, df, ano):
        """
        Salva DataFrame na camada Gold usando Delta Lake
        """
        delta_path = f"s3a://{self.minio_config.bucket_gold}/cvm/indicadores_graham"
        
        logger.info(f"  💾 Salvando em {delta_path}")
        
        try:
            # Tentar salvar com merge (upsert) se tabela já existe
            df.write.format("delta") \
                .mode("overwrite") \
                .option("overwriteSchema", "true") \
                .option("replaceWhere", f"ANO_REFERENCIA = {ano}") \
                .partitionBy("ANO_REFERENCIA") \
                .save(delta_path)
            
            logger.info(f"  ✅ Dados salvos com sucesso (partição: {ano})")
            
        except Exception as e:
            logger.warning(f"  ⚠️  Erro ao salvar com particionamento: {e}")
            logger.info("  🔄 Tentando salvar sem particionamento...")
            
            # Fallback: salvar sem particionamento
            df.write.format("delta") \
                .mode("overwrite") \
                .option("overwriteSchema", "true") \
                .save(delta_path)
            
            logger.info("  ✅ Dados salvos (modo fallback)")
    
    def _show_statistics(self, df, ano):
        """
        Exibe estatísticas do processamento
        """
        total = df.count()
        aprovados_f1 = df.filter(col("passou_filtro_1") == True).count()
        aprovados_f3 = df.filter(col("passou_filtro_3") == True).count()
        aprovados_f4 = df.filter(col("passou_filtro_4") == True).count()
        aprovados_todos = df.filter(col("passou_todos_filtros") == True).count()
        
        empresas_unicas = df.select("CNPJ_CIA").distinct().count()
        
        logger.info(f"""
            ╔═══════════════════════════════════════════════════════════╗
            ║ RESULTADO DO PROCESSAMENTO - ANO {ano}                    
            ╠═══════════════════════════════════════════════════════════╣
            ║ Total de registros: {total:>6}                            
            ║ Empresas únicas: {empresas_unicas:>6}                     
            ║                                                           
            ║ CRITÉRIOS DE BENJAMIN GRAHAM:                            
            ║ Passou Filtro 1 (LL > 0): {aprovados_f1:>6} ({aprovados_f1/total*100:>5.1f}%)      
            ║ Passou Filtro 3 (Div > 0): {aprovados_f3:>6} ({aprovados_f3/total*100:>5.1f}%)     
            ║ Passou Filtro 4 (LC >= 1): {aprovados_f4:>6} ({aprovados_f4/total*100:>5.1f}%)     
            ║                                                           
            ║ 🎯 Passou TODOS os filtros: {aprovados_todos:>6} ({aprovados_todos/total*100:>5.1f}%)       
            ╚═══════════════════════════════════════════════════════════╝
        """)
    
    def read_gold_table(self, ano: Optional[int] = None):
        """
        Lê a tabela Delta da camada Gold
        
        Args:
            ano: Ano específico (opcional, None = todos os anos)
            
        Returns:
            Spark DataFrame
        """
        delta_path = f"s3a://{self.minio_config.bucket_gold}/cvm/indicadores_graham"
        
        df = self.spark.read.format("delta").load(delta_path)
        
        if ano:
            df = df.filter(
                (col("ANO_REFERENCIA") == ano) | 
                (col("ANO_REFERENCIA") == str(ano))
            )
        
        return df


# Instanciar processador
processor = CVMGoldProcessor()

# Processar múltiplos anos
if __name__ == "__main__":
    anos = [2020, 2021, 2022, 2023, 2024]
    
    logger.info("=" * 70)
    logger.info("🚀 INICIANDO PIPELINE GOLD - INDICADORES DE GRAHAM")
    logger.info("=" * 70)
    
    for ano in anos:
        success = processor.process_year(ano)
        status = "✅" if success else "❌"
        logger.info(f"\n{status} Ano {ano} processado\n")
    
    logger.info("=" * 70)
    logger.info("🎉 PIPELINE GOLD CONCLUÍDO!")
    logger.info("=" * 70)

In [ ]:
from silver import CVMSilverProcessor
processor = CVMSilverProcessor()

df = processor.read_silver_table("DRE_con")

df.show()

In [ ]:
# silver.py
from datetime import datetime
import zipfile
from io import BytesIO
import pandas as pd
from minio import Minio
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, trim, when, lit, current_timestamp, rlike, lower, translate, ilike
from pyspark.sql.types import DoubleType, IntegerType,StringType
from config import MinIOConfig
import logging
from typing import List, Dict, Optional
from silver_quality import SilverQualityValidator, silver_processing_time
import time
import tempfile
import os
import chardet

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

#AJUSTAR DEPOIS QUE FINALIZAR TUDO --------------------------------
def normalize_text(column):
    """
    Normaliza texto para facilitar filtros e comparações.

    Etapas:
    - converte para minúsculo
    - remove acentos
    """

    return translate(
        lower(column),
        "áàãâäéèêëíìîïóòõôöúùûüç",
        "aaaaaeeeeiiiiooooouuuuc"
    )
#------------------------------------------------------------

class SilverProcessor:
    """Classe base para processar dados da camada Bronze → Silver"""
    
    def __init__(self, minio_config: MinIOConfig = None):
        # Isso é só guardar as informações
        self.minio_config = minio_config or MinIOConfig()
        # Isso é abrir a conexão de fato
        self.minio_client = Minio(
            self.minio_config.endpoint,
            access_key=self.minio_config.access_key,
            secret_key=self.minio_config.secret_key,
            secure=self.minio_config.secure
        )
        self.spark = self._create_spark_session()
        self._ensure_bucket_exists()
        # Inicializar o validador de qualidade - prometheus
        self.validator = SilverQualityValidator(self.spark)
        logger.info("✅ SilverQualityValidator inicializado")
    def _create_spark_session(self) -> SparkSession:
        """Cria sessão Spark com Delta Lake e S3 configurados"""
        
        # Configurar Spark Builder
        builder = (
            # Inicia o construtor da sessão Spark
            SparkSession.builder

                # Nome da aplicação exibida na interface do Spark
                .appName("CVM_Silver_Processing")

                # ------------------------------------------------------------
                # Dependências externas que o Spark precisa baixar automaticamente
                # ------------------------------------------------------------
                .config(
                    "spark.jars.packages",
                    "io.delta:delta-core_2.12:2.4.0"  # Só Delta
                )
                .config(
                    "spark.jars",
                    "/opt/spark-jars/hadoop-aws-3.3.4.jar,/opt/spark-jars/aws-java-sdk-bundle-1.12.262.jar"
                )
                .config(
                    "spark.driver.extraClassPath",
                    "/opt/spark-jars/hadoop-aws-3.3.4.jar:/opt/spark-jars/aws-java-sdk-bundle-1.12.262.jar"
                )
                .config(
                    "spark.executor.extraClassPath",
                    "/opt/spark-jars/hadoop-aws-3.3.4.jar:/opt/spark-jars/aws-java-sdk-bundle-1.12.262.jar"
                )

                # ------------------------------------------------------------
                # Ativação do Delta Lake no Spark
                # ------------------------------------------------------------

                # Liga extensões SQL específicas do Delta Lake
                .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")

                # Torna o catálogo Delta o padrão para criação/leitura de tabelas
                .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

                # ------------------------------------------------------------
                # Configurações para acessar o MinIO usando S3A
                # ------------------------------------------------------------

                # Endereço HTTP do serviço MinIO
                .config("spark.hadoop.fs.s3a.endpoint", f"http://{self.minio_config.endpoint}")

                # Access key (usuário público) do MinIO
                .config("spark.hadoop.fs.s3a.access.key", self.minio_config.access_key)

                # Secret key (senha) do MinIO
                .config("spark.hadoop.fs.s3a.secret.key", self.minio_config.secret_key)

                # Obrigatório para MinIO: usa estilo path, não VHost
                .config("spark.hadoop.fs.s3a.path.style.access", "true")

                # Define explicitamente o filesystem S3A correto
                .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

                # Desativa SSL porque MinIO normalmente roda em HTTP no ambiente local
                .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

                # Usa provedor simples de credenciais (ideal para MinIO)
                .config("spark.hadoop.fs.s3a.aws.credentials.provider",
                        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

                # ------------------------------------------------------------
                # Configurações de performance do Spark
                # ------------------------------------------------------------

                # Ativa otimização adaptativa → Spark ajusta planos de execução dinamicamente
                .config("spark.sql.adaptive.enabled", "true")

                # Número de partições usadas em operações como join e groupBy
                .config("spark.sql.shuffle.partitions", "8")

                # Tamanho máximo de cada partição de arquivo (128 MB)
                # Evita gerar muitos arquivos pequenos
                .config("spark.sql.files.maxPartitionBytes", 134217728)
        )
        
        # Adicionar Delta Lake
        spark = configure_spark_with_delta_pip(builder).getOrCreate()
        spark.sparkContext.setLogLevel("WARN")
        
        logger.info("✅ Spark Session criada com Delta Lake")
        return spark
    
    def _ensure_bucket_exists(self):
        """Garante que o bucket silver existe"""
        if not self.minio_client.bucket_exists(self.minio_config.bucket_silver):
            self.minio_client.make_bucket(self.minio_config.bucket_silver)
            logger.info(f"Bucket {self.minio_config.bucket_silver} criado")


class CVMSilverProcessor(SilverProcessor):
    """Processa dados da CVM da camada Bronze para Silver usando Delta Lake"""
    
    # Definir quais CSVs queremos extrair do ZIP
    CSV_TARGETS = {
        "composicao_capital": "dfp_cia_aberta_composicao_capital",  # Composição do Capital Social
        "DRE_con": "dfp_cia_aberta_DRE_con",    # DRE Consolidado
        "DFC_DMPL_con": "dfp_cia_aberta_DMPL_con",  # Fluxo de Caixa Método Direto
        "BPA_con": "dfp_cia_aberta_BPA_con",     # Balanço Patrimonial Ativo Consolidado
        "BPP_con": "dfp_cia_aberta_BPP_con"      # Balanço Patrimonial Passivo Consolidado
    }
    
    def process_year(self, ano: str) -> Dict[str, bool]:
        """
        Processa todos os CSVs de um ano específico
        
        Args:
            ano: Ano a processar (ex: "2023")
            
        Returns:
            Dicionário com status de processamento de cada CSV
        """
        results = {}
        
        try:
            logger.info(f"🔄 Iniciando processamento Silver para ano {ano}")
            
            # 1. Buscar ZIP da camada Bronze
            zip_path = self._find_latest_zip(ano)
            if not zip_path:
                logger.error(f"❌ ZIP não encontrado para ano {ano}")
                return results
            
            # 2. Baixar ZIP do MinIO
            zip_bytes = self._download_from_minio(zip_path)
            
            # 3. Extrair e processar cada CSV
            with zipfile.ZipFile(BytesIO(zip_bytes)) as zf:
                for csv_key, csv_prefix in self.CSV_TARGETS.items():
                    csv_filename = f"{csv_prefix}_{ano}.csv"
                    
                    if csv_filename in zf.namelist():
                        logger.info(f"📄 Processando {csv_filename}...")
                        success = self._process_csv(zf, csv_filename, csv_key, ano)
                        results[csv_key] = success
                    else:
                        logger.warning(f"⚠️  {csv_filename} não encontrado no ZIP")
                        results[csv_key] = False
            
            logger.info(f"✅ Processamento concluído para {ano}")
            return results
            
        except Exception as e:
            logger.error(f"❌ Erro ao processar ano {ano}: {e}")
            return results
    
    def _find_latest_zip(self, ano: str) -> Optional[str]:
        """Encontra o ZIP mais recente para um ano na camada Bronze"""
        prefix = f"gov_br_cvm/demonstracoes_financeiras_padronizadas/ano={ano}/"
        
        try:
            objects = self.minio_client.list_objects(
                self.minio_config.bucket_bronze,
                prefix=prefix,
                recursive=True
            )
            
            zip_files = [obj.object_name for obj in objects if obj.object_name.endswith('.zip')]
            
            if not zip_files:
                return None
            
            # Retornar o mais recente (último da lista ordenada)
            return sorted(zip_files)[-1]
            
        except Exception as e:
            logger.error(f"❌ Erro ao buscar ZIP: {e}")
            return None
    
    def _download_from_minio(self, object_path: str) -> bytes:
        """Baixa objeto do MinIO e retorna bytes"""
        response = self.minio_client.get_object(
            self.minio_config.bucket_bronze,
            object_path
        )
        data = response.read()
        response.close()
        response.release_conn()
        return data
    
    def _process_csv(self, zf: zipfile.ZipFile, csv_filename: str, csv_key: str, ano: str) -> bool:
        """
        Processa um CSV específico do ZIP e salva como Delta Lake.
        Executa uma única validação de qualidade após salvar.
        Args:
            zf: Objeto ZipFile já aberto
            csv_filename: Nome do arquivo CSV dentro do ZIP
            csv_key: Chave identificadora para aplicar filtros específicos
            ano: Ano de referência (para metadata e métricas)
        Returns: True se processado com sucesso, False caso contrário
        """
        tmp_path = None
        try:
             # 1. Extrair CSV do ZIP para arquivo temporário
            with tempfile.NamedTemporaryFile(mode='wb', delete=False, suffix='.csv') as tmp_file:
                tmp_file.write(zf.read(csv_filename))
                tmp_path = tmp_file.name

            # Detectar encoding do arquivo (muitas vezes é ISO-8859-1)
            with open(tmp_path, 'rb') as f:
                encoding = chardet.detect(f.read(100000))['encoding']
            
            # 2. Ler diretamente com Spark
            df_spark = (
                self.spark.read
                .option("header", "true")
                .option("sep", ";")
                .option("encoding", encoding)
                .option("inferSchema", "false")  # Manter tudo como string inicialmente
                .csv(tmp_path)
            )
            logger.info(f"  📊 Linhas lidas: {df_spark.count():,}")

            # 3. Aplicar transformações de limpeza
            df_clean = self._clean_dataframe(df_spark, csv_key)

            # 4. Adicionar colunas de metadata
            df_final = (
                df_clean
                .withColumn("ano_referencia", lit(int(ano)))
                .withColumn("data_processamento", current_timestamp())
                .withColumn("fonte", lit("CVM"))
                .withColumn("tipo_demonstracao", lit(csv_key))
            )

            # 5. Salvar como Delta Lake
            delta_path = f"s3a://{self.minio_config.bucket_silver}/cvm/{csv_key}/"
            (
                df_final.write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .option("replaceWhere", f"ano_referencia = {ano}")
                .partitionBy("ano_referencia")
                .save(delta_path)
            )
            logger.info(f"  ✅ Salvo em Delta: {delta_path}")
            logger.info(f"  📈 Total de registros: {df_final.count():,}")

            # 6. Validar qualidade e enviar métricas ao Prometheus (UMA VEZ)
            #    Usar self.validator — inicializado no __init__, não criar instância nova
            start_time = time.time()
            self.validator.validate_and_report_metrics(df_final, csv_key, ano)
            elapsed = time.time() - start_time
            silver_processing_time.labels(
                csv_type=csv_key,
                ano=ano
            ).observe(elapsed)
            logger.info(f"⏱️ Tempo de validação: {elapsed:.2f}s")

            return True

        except Exception as e:
            logger.error(f"  ❌ Erro ao processar {csv_filename}: {e}")
            return False
        
        finally:
        # Limpar arquivo temporário
            if tmp_path and os.path.exists(tmp_path):
                os.remove(tmp_path)
                
    def _clean_dataframe(self, df, csv_key: str):
        """
        Aplica transformações de limpeza e padronização no DataFrame
        Args:
            df: DataFrame original lido do CSV
            csv_key: Chave identificadora para aplicar filtros específicos
        Returns: DataFrame limpo e padronizado

        """
        df_clean = df

        # Filter por DS_CONTA — apenas para tipos relevantes
        # Localização: silver.py, dentro de _clean_dataframe()
        # SUBSTITUIR o bloco ds_conta_filters pelo código abaixo:

        ds_conta_filters = {
            # ================================================================
            # DRE — Demonstração do Resultado do Exercício
            # ================================================================
            # Critérios Graham que dependem da DRE:
            #   ✅ Não ter prejuízo → precisa de "lucro líquido"
            #   ✅ P/L → precisa de "lucro por ação" ou "lucro líquido" + nº ações
            #   ✅ Dividendos → precisa de "dividendos"
            #
            "DRE_con": r"lucro/prejuízo consolidado do período|lucro ou prejuízo líquido consolidado do período|por ação",
            #            ↑                  ↑              
            #            |                  |             
            #     Para saber se         Para calcular     
            #   houve prejuízo              o P/L               

            # ================================================================
            # BPA — Balanço Patrimonial Ativo (o que a empresa TEM)
            # ================================================================
            # Critério Graham que depende do BPA:
            #   ✅ Liquidez Corrente → precisa de "ativo circulante"
            #
            "BPA_con": r"ativo circulante",
            #            ↑
            #     LC = Ativo Circulante / Passivo Circulante

            # ================================================================
            # BPP — Balanço Patrimonial Passivo (o que a empresa DEVE)
            # ================================================================
            # Critérios Graham que dependem do BPP:
            #   ✅ Liquidez Corrente → precisa de "passivo circulante"
            #   ✅ P/VPA → precisa de "patrimônio líquido"
            #
            "BPP_con": r"passivo circulante|patrimônio líquido consolidado",
            #            ↑                   ↑
            #   LC = Ativo Circ /     P/VPA = Preço / (PL / nº ações)
            #        Passivo Circ

            # ================================================================
            # DFC_DMPL — Mutações do Patrimônio Líquido
            # ================================================================
            # Mantido igual ao original (filtro por COLUNA_DF, não DS_CONTA)
            # Sem alteração necessária aqui.
        }

        # Resultado esperado: ~30.000–50.000 registros (vs ~12.000 antes)
        # Motivo: adicionamos "lucro líquido" e "patrimônio líquido"

        # ================================================================
        # 💡 FILTROS PARA DESENVOLVIMENTO FUTURO (não implementados agora)
        # ================================================================
        # Os filtros abaixo seriam úteis para análises mais avançadas,
        # mas não são necessários para os critérios de Graham utilizados
        # neste TCC (Silva Júnior, 2020). Deixados como referência:
        #
        # "DRE_con": r"receita|lucro|ebitda|resultado|custos|despesas|margem"
        #   → Para calcular margens operacionais, EBITDA, ROE etc.
        #
        # "BPA_con": r"ativo|caixa|aplicações financeiras|contas a receber|
        #              estoques|imobilizado|intangível|investimentos"
        #   → Para análise completa do balanço patrimonial
        #
        # "BPP_con": r"passivo|fornecedores|empréstimos|financiamentos|
        #              debêntures|capital social|reservas"
        #   → Para análise de endividamento e estrutura de capital
        #
        # "DFC_DMPL_con": r"patrimônio líquido|capital|reservas|lucros acumulados"
        #   → Para análise de fluxo de caixa completo
        # ================================================================
        required_cols = ["DS_CONTA", "ORDEM_EXERC"]
        if csv_key in ds_conta_filters and all(col in df_clean.columns for col in required_cols):
            df_clean = df_clean.filter(
                lower(col("DS_CONTA")).rlike(ds_conta_filters[csv_key]) &
                lower(col("ORDEM_EXERC")).ilike("ÚLTIMO")
            )

        # Filter COLUNA_DF — apenas para DMPL
        if csv_key == "DFC_DMPL_con" and "COLUNA_DF" in df_clean.columns:
            df_clean = df_clean.filter(
                lower(col("COLUNA_DF")).rlike(r"patrimônio líquido") &
                lower(col("ORDEM_EXERC")).ilike("ÚLTIMO")
            )

        # Trimmar strings
        string_cols = [field.name for field in df.schema.fields
                    if isinstance(field.dataType, StringType)]
        for col_name in string_cols:
            df_clean = df_clean.withColumn(col_name, trim(col(col_name)))

        # Converter VL_CONTA com escala
        if "VL_CONTA" in df_clean.columns:
            df_clean = df_clean.withColumn(
                "VL_CONTA",
                    when(
                        lower(col("DS_CONTA")).like("%por ação%"),   
                        col("VL_CONTA").cast(DoubleType())        
                    )
                    .when(
                        col("ESCALA_MOEDA") == "MIL",
                        col("VL_CONTA").cast(DoubleType()) * 1000
                    )
                    .otherwise(col("VL_CONTA").cast(DoubleType()))
                )
            

        # Converter datas
        date_cols = ['DT_REFER', 'DT_INI_EXERC', 'DT_FIM_EXERC']
        for col_name in date_cols:
            if col_name in df_clean.columns:
                df_clean = df_clean.withColumn(
                    col_name,
                    to_date(col(col_name), "yyyy-MM-dd")
                )

        # Remover linhas completamente nulas
        df_clean = df_clean.dropna(how='all')

        # ⚠️ Guardar se DataFrame está vazio antes de retornar
        count = df_clean.count()
        if count == 0:
            logger.warning(f"⚠️ DataFrame vazio após filtros para {csv_key}!")
        else:
            logger.info(f"  ✅ {count:,} linhas após limpeza")

        return df_clean
        
    def read_silver_table(self, table_name: str, ano: Optional[int] = None):
        """
        Lê uma tabela Delta da camada Silver
        
        Args:
            table_name: Nome da tabela (BP_con, DRE_con, etc)
            ano: Ano específico (opcional, None = todos os anos)
            
        Returns:
            Spark DataFrame
        """
        delta_path = f"s3a://{self.minio_config.bucket_silver}/cvm/{table_name}"
        
        df = self.spark.read.format("delta").load(delta_path)
        
        if ano:
            df = df.filter(
                (col("ano_referencia") == ano) | 
                (col("ano_referencia") == str(ano))
            )
        
        return df


processor = CVMSilverProcessor()

# Processar múltiplos anos
if __name__ == "__main__":
    anos = ["2020", "2021", "2022", "2023", "2024","2025"]
    for ano in anos:
        results = processor.process_year(ano)
        print(f"\n📊 Resultados para {ano}:")
        for csv_key, success in results.items():
            status = "✅" if success else "❌"
            print(f"  {status} {csv_key}")
""" if __name__ == "__main__":
    #ler delta table
    for table in ["DRE_con", "BPA_con", "BPP_con", "DFC_DMPL_con"]:
        df = processor.read_silver_table(table, ano=2020)
        #converter para pandas
        psdf = df.to_pandas_on_spark()
        #salvar como excel
        psdf.to_excel(f"{table}_teste.xlsx", index=False) """

In [ ]:
import pandas
df=pandas.read_csv("dfp_cia_aberta_DRE_con_2020.csv", sep=";", encoding="ISO-8859-1")
test=1

In [ ]:
df_clean.filter(col("DS_CONTA").like("%por%")) \
        .select("DS_CONTA") \
        .distinct() \
        .show(truncate=False)